In [ ]:
%pip install -q monai nibabel

In [ ]:
import monai, torch, nibabel as nib, numpy as np
print("monai", monai.__version__, "| torch", torch.__version__, "| nibabel", nib.__version__)
print("cuda", torch.cuda.is_available())

In [ ]:
from pathlib import Path
from monai.apps import DecathlonDataset

ROOT = Path("/tmp/msd")
ROOT.mkdir(parents=True, exist_ok=True)
TASK, SEED, VAL_FRAC = "Task09_Spleen", 42, 0.2


def datalist(section: str) -> list[dict]:
    """File dicts {'image', 'label'} for one section; same seed for both sections."""
    ds = DecathlonDataset(
        root_dir=str(ROOT), task=TASK, section=section, transform=(),
        download=True, seed=SEED, val_frac=VAL_FRAC, cache_rate=0.0,
    )
    if section == "training":
        print(ds.get_properties(["numTraining", "numTest"]))
    return ds.data


train_files = datalist("training")
val_files = datalist("validation")
print(len(train_files), len(val_files))
print(train_files[0])

tr = {Path(d["image"]).name for d in train_files}
va = {Path(d["image"]).name for d in val_files}
assert not tr & va, tr & va
assert len(tr | va) == 41, len(tr | va)
for d in train_files + val_files:
    assert Path(d["image"]).name == Path(d["label"]).name, d
print("val:", sorted(va))

In [ ]:
d = train_files[0]
img, lab = nib.load(d["image"]), nib.load(d["label"])
print(Path(d["image"]).name)
print("shape", img.shape, "| on-disk dtype", img.get_data_dtype())
print("spacing mm", tuple(round(float(z), 3) for z in img.header.get_zooms()))
print("axcodes", nib.aff2axcodes(img.affine))
print("slope/inter", img.header.get_slope_inter())
print("affine\n", np.round(img.affine, 2))
x = img.get_fdata(dtype=np.float32)
y = lab.get_fdata(dtype=np.float32)
print("HU min/max", x.min(), x.max(), "| label values", np.unique(y))

In [ ]:
import pandas as pd

rows = []
for split, files in (("train", train_files), ("val", val_files)):
    for d in files:
        img, lab = nib.load(d["image"]), nib.load(d["label"])
        y = np.asarray(lab.dataobj)
        vals = set(np.unique(y).tolist())
        assert img.shape == lab.shape, d["image"]
        assert np.allclose(img.affine, lab.affine, atol=1e-3), d["image"]
        assert vals == {0, 1}, (d["image"], vals)
        zx, zy, zz = (float(z) for z in img.header.get_zooms())
        rows.append({
            "name": Path(d["image"]).name, "split": split,
            "nx": img.shape[0], "ny": img.shape[1], "nz": img.shape[2],
            "sx": zx, "sz": zz,
            "axcodes": "".join(nib.aff2axcodes(img.affine)),
            "spleen_ml": (y > 0).sum() * zx * zy * zz / 1000,
        })
vols = pd.DataFrame(rows)
print(vols.describe().round(2))
print(vols["axcodes"].value_counts())
print(vols.groupby("split")["spleen_ml"].describe().round(0))

In [ ]:
import matplotlib.pyplot as plt

case = train_files[0]
im = nib.load(case["image"]).get_fdata(dtype=np.float32)
lb = nib.load(case["label"]).get_fdata(dtype=np.float32)
assert im.shape == lb.shape
k = int(lb.sum(axis=(0, 1)).argmax())
win = np.clip((im[:, :, k] + 57) / (164 + 57), 0, 1)   # soft-tissue window -57..164 HU
mask = lb[:, :, k]
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(np.rot90(win), cmap="gray")
ax[0].set_title(f"{Path(case['image']).name} slice {k}")
ax[1].imshow(np.rot90(win), cmap="gray")
ax[1].imshow(np.rot90(np.ma.masked_where(mask == 0, mask)), cmap="autumn", alpha=0.5)
ax[1].set_title("label overlay")
for a in ax:
    a.axis("off")
plt.show()

In [ ]:
import torch
from monai.transforms import (
    Compose, CropForegroundd, EnsureChannelFirstd, LoadImaged, Orientationd,
    RandCropByPosNegLabeld, ScaleIntensityRanged, Spacingd,
)

KEYS = ["image", "label"]
PIXDIM = (1.5, 1.5, 2.0)
HU_MIN, HU_MAX = -57, 164
ROI = (96, 96, 96)

base = [
    LoadImaged(keys=KEYS),
    EnsureChannelFirstd(keys=KEYS),
    Orientationd(keys=KEYS, axcodes="RAS"),
    Spacingd(keys=KEYS, pixdim=PIXDIM, mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys="image", a_min=HU_MIN, a_max=HU_MAX, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=KEYS, source_key="image", allow_smaller=True),
]


def crop(pos: int, neg: int) -> RandCropByPosNegLabeld:
    """Random 96^3 crops: pos/neg = ratio of spleen-centred vs background-centred crops."""
    return RandCropByPosNegLabeld(
        keys=KEYS, label_key="label", spatial_size=ROI, pos=pos, neg=neg,
        num_samples=4, image_key="image", image_threshold=0,
    )


val_tf = Compose(base)
train_tf = Compose([*base, crop(1, 1)])

In [ ]:
voxel_ml = float(np.prod(PIXDIM)) / 1000
rows = []
for case in train_files + val_files:
    out = val_tf(case)
    assert out["image"].shape == out["label"].shape, case["image"]
    vals = set(torch.unique(out["label"]).tolist())
    assert vals == {0, 1}, (case["image"], vals)
    rows.append({
        "name": Path(case["image"]).name,
        "shape": tuple(out["image"].shape[1:]),
        "imin": float(out["image"].min()),
        "imax": float(out["image"].max()),
        "ml_after": float(out["label"].sum()) * voxel_ml,
    })
after = pd.DataFrame(rows).merge(vols[["name", "spleen_ml"]], on="name", how="left", validate="one_to_one")
assert after["spleen_ml"].notna().all()
after["diff_pct"] = 100 * (after["ml_after"] / after["spleen_ml"] - 1)
dims = np.array(after["shape"].tolist())
print("pixdim", tuple(round(float(p), 3) for p in out["image"].pixdim))
print("min dims", dims.min(axis=0), "| max dims", dims.max(axis=0))
print("intensity", after["imin"].min(), after["imax"].max())
print(f"volume diff %: mean {after['diff_pct'].mean():.2f}  max abs {after['diff_pct'].abs().max():.2f}")
print(after.loc[after["name"] == "spleen_28.nii.gz", ["shape", "ml_after", "spleen_ml"]])
print("axcodes after", nib.aff2axcodes(out["image"].affine.numpy()))

In [ ]:
import matplotlib.pyplot as plt

pre = val_tf(train_files[0])
im3, lb3 = pre["image"][0].numpy(), pre["label"][0].numpy()
k3 = int(lb3.sum(axis=(0, 1)).argmax())
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(np.rot90(im3[:, :, k3]), cmap="gray")
ax[0].set_title(f"after transforms, slice {k3}, shape {im3.shape}")
ax[1].imshow(np.rot90(im3[:, :, k3]), cmap="gray")
ax[1].imshow(np.rot90(np.ma.masked_where(lb3[:, :, k3] == 0, lb3[:, :, k3])), cmap="autumn", alpha=0.5)
for a in ax:
    a.axis("off")
plt.show()

In [ ]:
def spleen_hit_rate(pos: int, neg: int, n_calls: int = 25) -> float:
    """Fraction of crops that contain any spleen voxel (deterministic seed)."""
    c = crop(pos, neg)
    c.set_random_state(seed=0)
    hits = [float(s["label"].sum() > 0) for _ in range(n_calls) for s in c(pre)]
    return float(np.mean(hits))


one = crop(1, 1)(pre)
print(len(one), [tuple(s["image"].shape) for s in one])
print("pos1/neg1 hit rate", round(spleen_hit_rate(1, 1), 2))
print("pos0/neg1 hit rate", round(spleen_hit_rate(0, 1), 2))

In [ ]:
def spleen_frac(pos: int, neg: int, n_calls: int = 25) -> float:
    """Mean fraction of spleen voxels per crop (deterministic seed)."""
    c = crop(pos, neg)
    c.set_random_state(seed=0)
    return float(np.mean([s["label"].mean().item() for _ in range(n_calls) for s in c(pre)]))


print("whole volume", round(pre["label"].mean().item(), 4))
print("pos1/neg1   ", round(spleen_frac(1, 1), 4))
print("pos0/neg1   ", round(spleen_frac(0, 1), 4))

In [ ]:
from monai.metrics import DiceMetric
from monai.networks.utils import one_hot

dice_metric = DiceMetric(include_background=False, reduction="mean")


def dice_of(pred_lbl: torch.Tensor, true_lbl: torch.Tensor) -> float:
    """Dice of two (1, H, W, D) label volumes, through the metric used in validation."""
    out = dice_metric(y_pred=one_hot(pred_lbl[None], num_classes=2), y=one_hot(true_lbl[None], num_classes=2))
    return out.item()


gt = pre["label"]
empty = torch.zeros_like(gt)
print("perfect              ", dice_of(gt, gt))
print("empty pred           ", dice_of(empty, gt))
print("empty pred, accuracy ", round((empty == gt).float().mean().item(), 4))
print("shift 20 mm (10 sl.) ", round(dice_of(torch.roll(gt, 10, dims=-1), gt), 3))
print("empty pred, empty gt ", dice_of(empty, empty))
dice_metric.reset()

In [ ]:
from monai.data import DataLoader, Dataset
from monai.losses import DiceLoss
from monai.networks.layers import Norm
from monai.networks.nets import UNet


def build_unet() -> UNet:
    """3D U-Net from the MONAI spleen tutorial: 1 input channel, 2 classes."""
    return UNet(
        spatial_dims=3, in_channels=1, out_channels=2,
        channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
        num_res_units=2, norm=Norm.BATCH,
    )


probe = DataLoader(Dataset(train_files[:2], transform=train_tf), batch_size=2, shuffle=False, num_workers=0)
batch = next(iter(probe))
print("batch", tuple(batch["image"].shape), tuple(batch["label"].shape))

model = build_unet()
print("params", sum(p.numel() for p in model.parameters()))
loss_fn = DiceLoss(to_onehot_y=True, softmax=True)
x, y = batch["image"][:2], batch["label"][:2]
logits = model(x)
loss = loss_fn(logits, y)
loss.backward()
print("logits", tuple(logits.shape), "| loss", round(loss.item(), 3))
print("all grads present", all(p.grad is not None for p in model.parameters()))

In [ ]:
import time

from monai.data import CacheDataset, decollate_batch
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete
from monai.utils import set_determinism

assert torch.cuda.is_available(), "switch the accelerator to GPU"
device = torch.device("cuda")
print(torch.cuda.get_device_name(0))

SMOKE = False
MAX_EPOCHS = 4 if SMOKE else 600
VAL_INTERVAL = 2 if SMOKE else 5
assert MAX_EPOCHS % VAL_INTERVAL == 0, "last epoch must be a validation epoch"
SW_ROI = (160, 160, 160)
OUT = Path("/kaggle/working")
CKPT = OUT / ("smoke_unet.pt" if SMOKE else "spleen_unet_best.pt")
LAST = OUT / ("smoke_unet_last.pt" if SMOKE else "spleen_unet_last.pt")
HIST = OUT / ("smoke_history.csv" if SMOKE else "spleen_history.csv")
VAL_NAMES = [Path(d["image"]).name for d in val_files]

set_determinism(seed=0)
t_cache = time.time()
train_ds = CacheDataset(train_files, transform=train_tf, cache_rate=1.0, num_workers=4)
val_ds = CacheDataset(val_files, transform=val_tf, cache_rate=1.0, num_workers=4)
print(f"cache {time.time() - t_cache:.0f}s")
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=4)

model = build_unet().to(device)
loss_fn = DiceLoss(to_onehot_y=True, softmax=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-4)
post_pred = AsDiscrete(argmax=True, to_onehot=2)
post_label = AsDiscrete(to_onehot=2)
val_dice = DiceMetric(include_background=False, reduction="mean")


@torch.no_grad()
def validate() -> dict:
    """Whole-volume sliding-window Dice for each val volume, in loader (= val_files) order."""
    model.eval()
    scores = []
    for vb in val_loader:
        vx, vy = vb["image"].to(device), vb["label"].to(device)
        logits = sliding_window_inference(vx, SW_ROI, sw_batch_size=4, predictor=model)
        preds = [post_pred(p) for p in decollate_batch(logits)]
        labs = [post_label(t) for t in decollate_batch(vy)]
        scores.append(val_dice(y_pred=preds, y=labs).item())
    val_dice.reset()
    assert len(scores) == len(VAL_NAMES), (len(scores), len(VAL_NAMES))
    return dict(zip(VAL_NAMES, scores, strict=True))


history, best = [], -1.0
for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()
    model.train()
    losses = []
    for batch in train_loader:
        x, y = batch["image"].to(device), batch["label"].to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "train_sec": time.time() - t0}
    msg = f"epoch {epoch:3d}  loss {row['train_loss']:.4f}  {row['train_sec']:.0f}s"
    if epoch % VAL_INTERVAL == 0:
        t1 = time.time()
        per_vol = validate()
        row |= {"val_dice": float(np.mean(list(per_vol.values()))),
                "val_dice_min": float(min(per_vol.values())),
                "val_sec": time.time() - t1}
        msg += f"  | val dice {row['val_dice']:.4f} (min {row['val_dice_min']:.4f})  {row['val_sec']:.0f}s"
        if row["val_dice"] > best:
            best = row["val_dice"]
            torch.save({"state_dict": model.state_dict(), "epoch": epoch, "val_dice": best,
                        "per_vol": per_vol, "pixdim": PIXDIM, "hu_window": (HU_MIN, HU_MAX),
                        "roi": ROI}, CKPT)
            msg += "  *saved*"
    history.append(row)
    pd.DataFrame(history).to_csv(HIST, index=False)
    print(msg)

torch.save({"state_dict": model.state_dict(), "epoch": MAX_EPOCHS, "val_dice": row["val_dice"],
            "per_vol": per_vol, "pixdim": PIXDIM, "hu_window": (HU_MIN, HU_MAX), "roi": ROI}, LAST)
print(f"best {best:.4f} | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(sorted(p.name for p in OUT.iterdir()))